# Serialization and Batching

The previous section built a thread-safe asynchronous profiling pipeline:

```text
SyntheticEventSource
        ↓
Producer Thread
        ↓
BoundedQueue<ProfileEvent>
        ↓
Consumer Thread
        ↓
ConsoleSink
```

At this point, profiling events can move safely between threads.

However, the current pipeline still passes C++ objects directly:

```cpp
ProfileEvent
```

This works inside one process, but it is not sufficient for:

- network transmission
- writing compact profiling data to files
- sending data between processes
- sending data from an agent to a remote collector

Before profiling data can be transmitted, it must be converted into a portable byte representation.

This process is called:

```text
Serialization
```

---

## What Is Serialization?

Serialization converts an in-memory C++ object into a sequence of bytes.

For example, a profiling event such as:

```text
KernelEvent
timestamp = 1000
duration  = 200
device    = 0
kernel    = gemm_kernel
```

might be serialized into a text representation:

```text
KERNEL|1000|200|0|gemm_kernel
```

or eventually into a compact binary representation:

```text
[header][timestamp][duration][device][name_length][name_bytes]
```

The important idea is:

```text
C++ object
    ↓
serialization
    ↓
byte sequence
```

The receiver can later perform the reverse operation:

```text
byte sequence
    ↓
deserialization
    ↓
C++ object
```

---

## Why Serialization Is Necessary

A `ProfileEvent` contains C++ data structures such as:

```cpp
std::variant
std::string
std::uint64_t
```

These objects cannot simply be copied directly across a TCP connection.

For example, a `std::string` object contains implementation-specific internal state and usually references memory stored elsewhere.

Therefore, sending the raw memory occupied by:

```cpp
ProfileEvent
```

would not create a valid `ProfileEvent` on another machine.

Instead, we must define an explicit wire format.

Conceptually:

```text
ProfileEvent
      ↓
Serializer
      ↓
well-defined byte format
      ↓
TCP / file / database
```

---

## Existing Serializer Interface

The project already defines:

```cpp
class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};
```

and an initial implementation:

```cpp
SimpleTextSerializer
```

which converts profiling events into text-based byte sequences.

For example:

```text
KernelEvent
```

becomes:

```text
KERNEL|1000|200|0|gemm_kernel
```

and:

```text
MemoryEvent
```

becomes:

```text
MEMORY|1500|0|4096|H2D
```

This is useful because the format is easy to inspect and debug.

---

## From Individual Events to Batches

Sending every profiling event independently is simple, but it may be inefficient.

Suppose the profiler generates:

```text
100,000 events / second
```

If every event causes an independent network operation:

```text
event 1 → send()
event 2 → send()
event 3 → send()
...
```

the system may perform a very large number of:

```text
system calls
network writes
packet operations
```

Instead, multiple serialized events can be grouped into a batch:

```text
Event 1 ─┐
Event 2 ─┤
Event 3 ─┤
Event 4 ─┘
          ↓
        Batch
          ↓
      one send()
```

This reduces per-event overhead.

---

## Why Batching Helps Throughput

Without batching:

```text
Event
  ↓
serialize
  ↓
send()

Event
  ↓
serialize
  ↓
send()

Event
  ↓
serialize
  ↓
send()
```

With batching:

```text
Event
Event
Event
Event
  ↓
serialize
  ↓
Batch Buffer
  ↓
send()
```

A larger batch can improve:

- network throughput
- syscall efficiency
- serialization efficiency
- CPU utilization

because the fixed overhead of each send operation is amortized over multiple events.

---

## The Batching Tradeoff

Batching is not free.

A larger batch improves efficiency, but an event may need to wait longer before being sent.

For example:

```text
batch size = 1
```

means:

```text
very low batching delay
higher per-event overhead
```

while:

```text
batch size = 100
```

may mean:

```text
better throughput
lower overhead per event
higher waiting latency
```

Therefore, batching introduces a fundamental tradeoff:

```text
larger batch
    ↓
higher throughput
but potentially
higher latency
```

This tradeoff is directly relevant to profiling infrastructure because profiling systems may need both:

```text
high event throughput
```

and:

```text
timely visibility of performance data
```

---

## Target Architecture

The pipeline will evolve from:

```text
ProfileEvent
    ↓
ConsoleSink
```

into:

```text
ProfileEvent
    ↓
Serializer
    ↓
Serialized Event
    ↓
Batch Builder
    ↓
Byte Buffer
    ↓
Transport
```

Eventually, this will become:

```text
Synthetic / GPU Event Source
          ↓
     Producer Thread
          ↓
BoundedQueue<ProfileEvent>
          ↓
     Consumer Thread
          ↓
       Serializer
          ↓
      Batch Builder
          ↓
       TCP Agent
          ↓
        Network
          ↓
       Collector
```

---

## Message Framing

Once several serialized events are placed into a byte stream, the receiver must know where one message ends and the next begins.

TCP provides:

```text
a byte stream
```

not:

```text
a stream of complete application messages
```

For example, sending:

```text
EVENT_A
EVENT_B
```

does not guarantee that the receiver will obtain exactly two matching `recv()` calls.

The receiver might instead observe:

```text
recv 1 → part of EVENT_A
recv 2 → rest of EVENT_A + part of EVENT_B
recv 3 → rest of EVENT_B
```

Therefore, the application must define message boundaries.

This is called:

```text
framing
```

Common approaches include:

```text
delimiter-based framing
length-prefix framing
fixed-size messages
```

For this project, length-prefix framing will eventually be a useful design because profiling events can have variable lengths.

Conceptually:

```text
[length][payload]
```

For example:

```text
[29][KERNEL|1000|200|0|gemm_kernel]
```

The receiver first reads the length and then knows exactly how many payload bytes belong to that message.

---

## Goals of This Section

By the end of this notebook, the following questions should be answerable:

- What is serialization?
- Why can a C++ object not simply be sent directly over TCP?
- Why use `std::vector<std::byte>` as a serialized representation?
- What is the difference between text and binary serialization?
- What is batching?
- Why can batching improve throughput?
- What latency cost can batching introduce?
- Why does TCP require application-level message framing?
- What is length-prefix framing?
- How should serialized profiling events be prepared for the networking stage?

The implementation will start with the existing `SimpleTextSerializer`, then build batching and framing around it before introducing TCP transport.

## Step1: Standalone Setup

This notebook is designed to run independently in a fresh Google Colab runtime.

The setup installs and verifies the minimal development tools required for the C++20 serialization and batching pipeline.

Required tools:

- GCC / G++ with C++20 support
- CMake
- Ninja
- Git

This notebook does not require a GPU.

The focus is on host-side data processing:

- profiling event serialization
- conversion from C++ objects to byte buffers
- text-based wire formats
- message framing
- length-prefix encoding
- batching multiple profiling events
- throughput versus latency tradeoffs
- preparation for later TCP transport

The notebook recreates only the minimal shared project files needed from the previous sections so that it can be executed independently without relying on an earlier Colab runtime.

In [1]:
!apt-get update -qq
!apt-get install -y -qq build-essential g++ cmake ninja-build

!mkdir -p distributed-profiler/include
!mkdir -p distributed-profiler/src
!mkdir -p distributed-profiler/tests
!mkdir -p distributed-profiler/benchmarks

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package ninja-build.
(Reading database ... 122815 files and directories currently installed.)
Preparing to unpack .../ninja-build_1.11.1-2_amd64.deb ...
Unpacking ninja-build (1.11.1-2) ...
Setting up ninja-build (1.11.1-2) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
%%writefile distributed-profiler/include/profile_event.hpp

#pragma once

#include <cstdint>
#include <string>
#include <variant>

struct KernelEvent {
    std::uint64_t timestamp_ns;
    std::uint64_t duration_ns;
    std::uint32_t device_id;
    std::string kernel_name;
};

struct MemoryEvent {
    std::uint64_t timestamp_ns;
    std::uint32_t device_id;
    std::uint64_t bytes;
    std::string operation;
};

using ProfileEvent = std::variant<KernelEvent, MemoryEvent>;

Writing distributed-profiler/include/profile_event.hpp


In [3]:
%%writefile distributed-profiler/include/serializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <vector>

class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};

Writing distributed-profiler/include/serializer.hpp


In [4]:
%%writefile distributed-profiler/include/simple_text_serializer.hpp

#pragma once

#include "serializer.hpp"

class SimpleTextSerializer : public ISerializer {
public:
    std::vector<std::byte>
    serialize(const ProfileEvent& event) const override;
};

Writing distributed-profiler/include/simple_text_serializer.hpp


In [5]:
%%writefile distributed-profiler/src/simple_text_serializer.cpp

#include "simple_text_serializer.hpp"

#include <string>
#include <type_traits>
#include <variant>

std::vector<std::byte>
SimpleTextSerializer::serialize(const ProfileEvent& event) const {
    std::string text;

    std::visit(
        [&](const auto& e) {
            using T = std::decay_t<decltype(e)>;

            if constexpr (std::is_same_v<T, KernelEvent>) {
                text =
                    "KERNEL|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.duration_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    e.kernel_name;
            }
            else if constexpr (std::is_same_v<T, MemoryEvent>) {
                text =
                    "MEMORY|" +
                    std::to_string(e.timestamp_ns) + "|" +
                    std::to_string(e.device_id) + "|" +
                    std::to_string(e.bytes) + "|" +
                    e.operation;
            }
        },
        event
    );

    std::vector<std::byte> bytes;
    bytes.reserve(text.size());

    for (char c : text) {
        bytes.push_back(static_cast<std::byte>(c));
    }

    return bytes;
}

Writing distributed-profiler/src/simple_text_serializer.cpp


In [6]:
%%writefile distributed-profiler/tests/test_serialization_environment.cpp

#include "profile_event.hpp"
#include "simple_text_serializer.hpp"

#include <cstddef>
#include <iostream>

int main() {
    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };

    ProfileEvent event = kernel;

    SimpleTextSerializer serializer;

    auto bytes = serializer.serialize(event);

    std::cout << "Serialized bytes: ";

    for (std::byte b : bytes) {
        std::cout << static_cast<char>(b);
    }

    std::cout << '\n';

    return 0;
}

Writing distributed-profiler/tests/test_serialization_environment.cpp


In [7]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/tests/test_serialization_environment.cpp \
    -o test_serialization_environment

In [8]:
!./test_serialization_environment

Serialized bytes: KERNEL|1000|200|0|gemm_kernel


## Step 2 — Single-Event Serialization

The goal of this step is to validate the basic serialization path for one profiling event at a time.

The input is a C++ profiling object:

```cpp
ProfileEvent
```

and the output is a byte buffer:

```cpp
std::vector<std::byte>
```

The data flow is:

```text
ProfileEvent
    ↓
ISerializer
    ↓
SimpleTextSerializer
    ↓
std::vector<std::byte>
```

---

### Why Serialize a Single Event First?

Before introducing batching, framing, or networking, the serialization of one event should be validated independently.

This isolates the basic transformation:

```text
C++ object
    ↓
serialized representation
```

If this step is incorrect, later batching and TCP transport will also be incorrect.

---

### Event Types

The current profiling model contains two event types:

```cpp
KernelEvent
```

and:

```cpp
MemoryEvent
```

They are represented through:

```cpp
using ProfileEvent =
    std::variant<KernelEvent, MemoryEvent>;
```

The serializer must therefore inspect which event type is currently stored inside the `std::variant`.

This is done using:

```cpp
std::visit(...)
```

---

### Kernel Event Format

A `KernelEvent` contains:

```text
timestamp_ns
duration_ns
device_id
kernel_name
```

For example:

```text
timestamp_ns = 1000
duration_ns  = 200
device_id    = 0
kernel_name  = gemm_kernel
```

The text serializer converts it into:

```text
KERNEL|1000|200|0|gemm_kernel
```

The fields are separated by:

```text
|
```

This provides a simple human-readable format that is easy to inspect during development.

---

### Memory Event Format

A `MemoryEvent` contains:

```text
timestamp_ns
device_id
bytes
operation
```

For example:

```text
timestamp_ns = 1500
device_id    = 0
bytes        = 4096
operation    = H2D
```

The serialized representation becomes:

```text
MEMORY|1500|0|4096|H2D
```

---

## Serializer Interface

The project uses an interface:

```cpp
class ISerializer {
public:
    virtual ~ISerializer() = default;

    virtual std::vector<std::byte>
    serialize(const ProfileEvent& event) const = 0;
};
```

This allows different serialization strategies to be introduced later without changing the rest of the pipeline.

For example:

```text
ISerializer
   ├── SimpleTextSerializer
   ├── BinarySerializer
   └── Future compact serializer
```

The profiling pipeline only depends on:

```cpp
ISerializer
```

instead of depending directly on one specific encoding format.

---

## Why Use `std::vector<std::byte>`?

The serializer returns:

```cpp
std::vector<std::byte>
```

rather than:

```cpp
std::string
```

because the final serialized representation is intended for systems-level data movement.

Later, the same byte buffer can be used for:

```text
network transmission
file writing
batch construction
binary protocols
```

Using bytes makes the interface independent of whether the underlying format is:

```text
text
binary
compressed
```

---

## Text Serialization Strategy

The current serializer first constructs a temporary:

```cpp
std::string
```

For example:

```text
KERNEL|1000|200|0|gemm_kernel
```

Then every character is converted into:

```cpp
std::byte
```

and stored in:

```cpp
std::vector<std::byte>
```

Conceptually:

```text
ProfileEvent
    ↓
std::visit()
    ↓
build text string
    ↓
convert characters to bytes
    ↓
std::vector<std::byte>
```

---

## Expected Validation

This step should test at least one `KernelEvent` and one `MemoryEvent`.

Expected kernel output:

```text
KERNEL|1000|200|0|gemm_kernel
```

Expected memory output:

```text
MEMORY|1500|0|4096|H2D
```

The test should confirm that the serialized bytes exactly match the expected representation.

---

## Why Start with a Text Format?

A text format is not necessarily the most compact representation, but it is useful during early development because it is:

- easy to read
- easy to debug
- easy to print
- easy to validate manually
- independent of platform-specific object layout

Later, a binary format can be introduced if lower overhead or smaller message size is required.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- What does serialization do?
- Why is `ProfileEvent` converted into bytes?
- How does `std::visit()` help serialize a `std::variant`?
- Why does the project use an `ISerializer` interface?
- Why return `std::vector<std::byte>` instead of a C++ object?
- What is the serialized format for `KernelEvent`?
- What is the serialized format for `MemoryEvent`?
- Why is a text format useful during initial development?

In [21]:
%%writefile distributed-profiler/include/deserializer.hpp

#pragma once

#include "profile_event.hpp"

#include <cstddef>
#include <optional>
#include <vector>

class IDeserializer {
public:
    // Add a virtual destructor.
    virtual ~IDeserializer() = default;


    // Declare a pure virtual deserialize() function.
    //
    // Input:
    //   const std::vector<std::byte>&
    //
    // Output:
    //   std::optional<ProfileEvent>
    //
    // This function should not modify the deserializer object.
    virtual std::optional<ProfileEvent>
    deserialize(
        const std::vector<std::byte>& data
    ) const = 0;
};

Overwriting distributed-profiler/include/deserializer.hpp


In [22]:
%%writefile distributed-profiler/include/simple_text_deserializer.hpp

#pragma once

#include "deserializer.hpp"

class SimpleTextDeserializer : public IDeserializer {
public:
    std::optional<ProfileEvent>
    deserialize(
        const std::vector<std::byte>& data
    ) const override;
};

Overwriting distributed-profiler/include/simple_text_deserializer.hpp


In [27]:
%%writefile distributed-profiler/src/simple_text_deserializer.cpp

#include "simple_text_deserializer.hpp"

#include <cstddef>
#include <sstream>
#include <string>
#include <vector>

std::optional<ProfileEvent>
SimpleTextDeserializer::deserialize(
    const std::vector<std::byte>& data
) const {

    // Convert std::vector<std::byte> into std::string.
    //
    // Example idea:
    //
    // std::string text;
    // for (...) {
    //     ...
    // }
    std::string text;
    text.reserve(data.size());

    for(std::byte b : data){
        text.push_back(static_cast<char>(b));
    }

    std::vector<std::string> tokens;

    std::stringstream ss(text);
    std::string token;


    // Split the text using "|" as the delimiter.
    //
    // Store fields in:
    //
    // std::vector<std::string> tokens;
    while (std::getline(ss, token, '|')) {
        tokens.push_back(token);
    }


    // If tokens is empty:
    //
    // return std::nullopt;
    if (tokens.empty()) {
        return std::nullopt;
    }

    // Check tokens[0].
    //
    // If:
    //
    // tokens[0] == "KERNEL"
    //
    // validate that the correct number of fields exists.


    // Convert kernel numeric fields:
    //
    // timestamp_ns
    // duration_ns
    // device_id
    //
    // Then construct a KernelEvent.

    // Return the KernelEvent as ProfileEvent.

    // Otherwise, if:
    //
    // tokens[0] == "MEMORY"
    //
    // validate field count.

    // Convert:
    //
    // timestamp_ns
    // device_id
    // bytes
    //
    // Then construct a MemoryEvent.


    // Return the MemoryEvent as ProfileEvent.

    // If the event type is unknown:
    //
    // return std::nullopt;
        try {
        if (tokens[0] == "KERNEL") {
            if (tokens.size() != 5) {
                return std::nullopt;
            }

            std::uint64_t timestamp_ns =
                std::stoull(tokens[1]);

            std::uint64_t duration_ns =
                std::stoull(tokens[2]);

            std::uint32_t device_id =
                static_cast<std::uint32_t>(
                    std::stoul(tokens[3])
                );

            std::string kernel_name =
                tokens[4];

            KernelEvent event{
                timestamp_ns,
                duration_ns,
                device_id,
                kernel_name
            };

            return ProfileEvent{event};
        }

        if (tokens[0] == "MEMORY") {
            if (tokens.size() != 5) {
                return std::nullopt;
            }

            std::uint64_t timestamp_ns =
                std::stoull(tokens[1]);

            std::uint32_t device_id =
                static_cast<std::uint32_t>(
                    std::stoul(tokens[2])
                );

            std::uint64_t bytes =
                std::stoull(tokens[3]);

            std::string operation =
                tokens[4];

            MemoryEvent event{
                timestamp_ns,
                device_id,
                bytes,
                operation
            };

            return ProfileEvent{event};
        }

        return std::nullopt;}


    // Consider invalid numeric conversions.
    //
    // For now, you can use try/catch around:
    //
    // std::stoull(...)
    // std::stoul(...)
    //
    // and return std::nullopt if conversion fails.
    catch (...) {
        return std::nullopt;
    }
}

Overwriting distributed-profiler/src/simple_text_deserializer.cpp


In [28]:
%%writefile distributed-profiler/tests/test_round_trip_serialization.cpp

#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"
#include "simple_text_serializer.hpp"

#include <iostream>
#include <type_traits>
#include <variant>

int main() {
    SimpleTextSerializer serializer;
    SimpleTextDeserializer deserializer;

    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };

    ProfileEvent kernel_event = kernel;

    auto kernel_bytes =
        serializer.serialize(kernel_event);

    auto reconstructed_kernel =
        deserializer.deserialize(kernel_bytes);

    if (!reconstructed_kernel.has_value()) {
        std::cerr
            << "Kernel deserialization failed\n";
        return 1;
    }

    bool kernel_passed = false;

    std::visit(
        [&](const auto& event) {
            using T =
                std::decay_t<decltype(event)>;

            if constexpr (
                std::is_same_v<T, KernelEvent>
            ) {
                kernel_passed =
                    event.timestamp_ns == 1000 &&
                    event.duration_ns == 200 &&
                    event.device_id == 0 &&
                    event.kernel_name == "gemm_kernel";
            }
        },
        *reconstructed_kernel
    );

    if (!kernel_passed) {
        std::cerr
            << "Kernel round-trip validation failed\n";
        return 1;
    }

    std::cout
        << "Kernel round-trip passed\n";


    MemoryEvent memory{
        1500,
        0,
        4096,
        "H2D"
    };

    ProfileEvent memory_event = memory;

    auto memory_bytes =
        serializer.serialize(memory_event);

    auto reconstructed_memory =
        deserializer.deserialize(memory_bytes);

    if (!reconstructed_memory.has_value()) {
        std::cerr
            << "Memory deserialization failed\n";
        return 1;
    }

    bool memory_passed = false;

    std::visit(
        [&](const auto& event) {
            using T =
                std::decay_t<decltype(event)>;

            if constexpr (
                std::is_same_v<T, MemoryEvent>
            ) {
                memory_passed =
                    event.timestamp_ns == 1500 &&
                    event.device_id == 0 &&
                    event.bytes == 4096 &&
                    event.operation == "H2D";
            }
        },
        *reconstructed_memory
    );

    if (!memory_passed) {
        std::cerr
            << "Memory round-trip validation failed\n";
        return 1;
    }

    std::cout
        << "Memory round-trip passed\n";

    return 0;
}

Overwriting distributed-profiler/tests/test_round_trip_serialization.cpp


In [29]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/tests/test_round_trip_serialization.cpp \
    -o test_round_trip_serialization

In [30]:
!./test_round_trip_serialization

Kernel round-trip passed
Memory round-trip passed


## Step 4 — Length-Prefix Framing

The previous step validated the complete serialization round trip:

```text
ProfileEvent
    ↓
serialize()
    ↓
std::vector<std::byte>
    ↓
deserialize()
    ↓
ProfileEvent
```

The next step is to prepare serialized profiling events for transport.

The main problem is that a transport such as TCP provides:

```text
a continuous byte stream
```

rather than:

```text
one complete application message per read
```

Therefore, the receiver needs a way to determine where each serialized event begins and ends.

This is called:

```text
Message Framing
```

---

## Why Framing Is Necessary

Suppose the sender has two serialized events:

```text
KERNEL|1000|200|0|gemm_kernel
```

and:

```text
MEMORY|1500|0|4096|H2D
```

The sender may conceptually call:

```text
send(event_1)
send(event_2)
```

However, the receiver is not guaranteed to receive them in exactly two matching reads.

The receiver might observe:

```text
read 1:
KERNEL|1000|200

read 2:
|0|gemm_kernelMEMORY|1500

read 3:
|0|4096|H2D
```

The byte order is preserved, but the original application-level message boundaries are not.

Therefore, the protocol itself must define those boundaries.

---

## Length-Prefix Framing

A common solution is:

```text
[length][payload]
```

The frame begins with the size of the payload.

For example:

```text
payload:
KERNEL|1000|200|0|gemm_kernel
```

becomes conceptually:

```text
[PAYLOAD_LENGTH][KERNEL|1000|200|0|gemm_kernel]
```

The receiver first reads the length.

Then it knows exactly how many bytes belong to the payload.

---

## Frame Structure

For this project, a frame will conceptually contain:

```text
+------------------+----------------------+
| Payload Length   | Serialized Payload   |
+------------------+----------------------+
```

where:

```text
Payload Length
```

describes the number of bytes in:

```text
Serialized Payload
```

For example:

```text
[31][KERNEL|1000|200|0|gemm_kernel]
```

The exact binary representation of the length field will be defined in the implementation.

A fixed-width integer such as:

```cpp
std::uint32_t
```

is a practical choice for the payload length.

---

## Why Use a Fixed-Width Length Field?

Using:

```cpp
std::uint32_t
```

gives the frame header a predictable size:

```text
4 bytes
```

Conceptually:

```text
4-byte length
+
N-byte payload
```

This makes parsing straightforward.

The receiver can:

```text
1. read 4 bytes
2. decode payload length
3. read exactly N bytes
4. reconstruct one message
```

---

## Framing Data Flow

The sender-side data flow becomes:

```text
ProfileEvent
    ↓
serialize()
    ↓
payload bytes
    ↓
frame()
    ↓
[length][payload]
```

The receiver performs the reverse operation:

```text
[length][payload]
    ↓
unframe()
    ↓
payload bytes
    ↓
deserialize()
    ↓
ProfileEvent
```

---

## Framing Is Separate from Serialization

Serialization and framing solve different problems.

Serialization answers:

```text
How do I represent a ProfileEvent as bytes?
```

Framing answers:

```text
How do I identify one serialized message inside a byte stream?
```

Therefore:

```text
ProfileEvent
    ↓
Serialization
    ↓
Payload
    ↓
Framing
    ↓
Transport-ready message
```

These should remain separate components.

---

## Target Framing API

A simple framing interface can conceptually provide:

```cpp
std::vector<std::byte>
frame(const std::vector<std::byte>& payload);
```

and:

```cpp
std::optional<std::vector<std::byte>>
unframe(const std::vector<std::byte>& frame);
```

The `frame()` function adds the length prefix.

The `unframe()` function validates the length and extracts the original payload.

---

## Basic Validation

The first test should use one profiling event.

For example:

```text
KernelEvent
    ↓
serialize()
    ↓
KERNEL|1000|200|0|gemm_kernel
    ↓
frame()
    ↓
[length][payload]
    ↓
unframe()
    ↓
KERNEL|1000|200|0|gemm_kernel
    ↓
deserialize()
    ↓
KernelEvent
```

The reconstructed event should match the original event.

---

## Invalid Frame Handling

The frame parser should also detect malformed input.

For example:

```text
length says 100 bytes
```

but the frame only contains:

```text
20 payload bytes
```

This frame is incomplete and should not be accepted.

Likewise, a frame with an invalid or inconsistent header should be rejected.

For the first implementation, failure can be represented using:

```cpp
std::optional<std::vector<std::byte>>
```

where:

```text
valid frame
    ↓
payload bytes
```

and:

```text
invalid frame
    ↓
std::nullopt
```

---

## Preparing for Batching

Once one frame can be created and parsed correctly, multiple frames can be placed into one larger buffer:

```text
[frame 1][frame 2][frame 3][frame 4]
```

This will become the basis of the batching stage.

The receiver can process the batch sequentially:

```text
read frame length
    ↓
extract payload
    ↓
move to next frame
    ↓
repeat
```

Therefore, framing is the bridge between:

```text
single-event serialization
```

and:

```text
multi-event batching
```

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- Why does TCP not preserve application-level message boundaries?
- What is message framing?
- What problem does length-prefix framing solve?
- Why use a fixed-width integer such as `std::uint32_t` for the length field?
- What is the difference between serialization and framing?
- How does the receiver determine the payload size?
- How should an incomplete or malformed frame be handled?
- How does framing prepare the system for batching and TCP transport?

In [31]:
%%writefile distributed-profiler/include/frame_codec.hpp

#pragma once

#include <cstddef>
#include <optional>
#include <vector>

class FrameCodec {
public:
    // Add a function:
    //
    // std::vector<std::byte>
    // frame(const std::vector<std::byte>& payload) const;
    //
    // It should:
    // - take a payload
    // - prepend a fixed-size length field
    // - return the complete frame
    std::vector<std::byte>
    frame(const std::vector<std::byte>& payload) const;


    // Add a function:
    //
    // std::optional<std::vector<std::byte>>
    // unframe(const std::vector<std::byte>& frame) const;
    //
    // It should:
    // - read the length field
    // - validate the frame size
    // - extract and return the payload
    // - return std::nullopt if the frame is malformed
    std::optional<std::vector<std::byte>>
    unframe(const std::vector<std::byte>& frame) const;
};

Writing distributed-profiler/include/frame_codec.hpp


In [32]:
%%writefile distributed-profiler/src/frame_codec.cpp

#include "frame_codec.hpp"

#include <cstdint>
#include <cstring>

/*
frame():

payload
   ↓
get payload.size()
   ↓
store size as uint32_t
   ↓
allocate:
4 bytes + payload bytes
   ↓
copy size into first 4 bytes
   ↓
copy payload after that


+--------------------+----------------------+
| uint32_t length    | payload              |
| 4 bytes            | N bytes              |
+--------------------+----------------------+

unframe()
frame
   ↓
check at least 4 bytes
   ↓
read uint32_t length
   ↓
calculate expected total size
   ↓
validate size
   ↓
extract payload

 */

std::vector<std::byte>
FrameCodec::frame(
    const std::vector<std::byte>& payload
) const {
    // Store payload.size() in a std::uint32_t.
    std::uint32_t payload_size =
        static_cast<std::uint32_t>(payload.size());

    // Create an output vector.
    //
    // Total size should be:
    //
    // sizeof(std::uint32_t) + payload.size()
    std::vector<std::byte> framed_data(
        sizeof(std::uint32_t) + payload.size()
    );


    // Append the 4-byte length prefix.
    std::memcpy(
        framed_data.data(),
        &payload_size,
        sizeof(std::uint32_t)
    );


    // Append all payload bytes.
    std::memcpy(
        framed_data.data() + sizeof(std::uint32_t),
        payload.data(),
        payload.size()
    );

    // Return the complete frame.
    return framed_data;
}


std::optional<std::vector<std::byte>>
FrameCodec::unframe(
    const std::vector<std::byte>& frame
) const {
    // First check that frame contains at least
    // sizeof(std::uint32_t) bytes.
    //
    // Otherwise return std::nullopt.
    if (frame.size() < sizeof(std::uint32_t)) {
        return std::nullopt;
    }

    // Read the first 4 bytes into a std::uint32_t
    // representing payload_size.
    std::uint32_t payload_size = 0;


    // Compute the expected total frame size:
    //
    // sizeof(std::uint32_t) + payload_size
    std::memcpy(
        &payload_size,
        frame.data(),
        sizeof(std::uint32_t)
    );


    // Validate that frame.size() matches the
    // expected total size.
    //
    // If not:
    // return std::nullopt;
    std::size_t expected_size =
        sizeof(std::uint32_t) +
        static_cast<std::size_t>(payload_size);

    if (frame.size() != expected_size) {
        return std::nullopt;
    }

    // Extract payload bytes into a new vector.
    std::vector<std::byte> payload(
        frame.begin() + sizeof(std::uint32_t),
        frame.end()
    );

    // Return the payload.
        return payload;
}

Writing distributed-profiler/src/frame_codec.cpp


In [33]:
%%writefile distributed-profiler/tests/test_length_prefix_framing.cpp

#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"
#include "simple_text_serializer.hpp"

#include <iostream>

int main() {
    SimpleTextSerializer serializer;
    SimpleTextDeserializer deserializer;
    FrameCodec codec;

    // Create:
    //
    // KernelEvent{
    //     1000,
    //     200,
    //     0,
    //     "gemm_kernel"
    // }
    KernelEvent kernel{
        1000,
        200,
        0,
        "gemm_kernel"
    };


    // Store it inside ProfileEvent.
    ProfileEvent event = kernel;

    // Serialize the event.
    auto payload =
        serializer.serialize(event);

    // Frame the serialized payload.
    auto framed =
        codec.frame(payload);

    // Print:
    //
    // payload size
    // frame size
    //
    // The frame should be exactly 4 bytes larger
    // than the payload.
    std::cout
        << "Payload size: "
        << payload.size()
        << " bytes\n";

    std::cout
        << "Frame size: "
        << framed.size()
        << " bytes\n";
        if (framed.size() !=
        payload.size() + sizeof(std::uint32_t)) {

        std::cerr
            << "Frame size validation failed\n";

        return 1;
    }

    // Unframe the frame.
    auto extracted_payload =
        codec.unframe(framed);



    // Check whether unframing succeeded.
    //
    // If not:
    // print error
    // return 1;
    if (!extracted_payload.has_value()) {
        std::cerr
            << "Unframing failed\n";

        return 1;
    }

    // Verify that the extracted payload is equal
    // to the original serialized payload.
    if (*extracted_payload != payload) {
        std::cerr
            << "Extracted payload does not match original payload\n";

        return 1;
    }

    // Deserialize the extracted payload.
    auto reconstructed_event =
        deserializer.deserialize(*extracted_payload);


    // Verify deserialization succeeded.
    if (!reconstructed_event.has_value()) {
        std::cerr
            << "Deserialization failed\n";

        return 1;
    }

    // Print something like:
    //
    // Length-prefix framing test passed
    std::cout
        << "Length-prefix framing test passed\n";


    return 0;
}

Writing distributed-profiler/tests/test_length_prefix_framing.cpp


In [34]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/tests/test_length_prefix_framing.cpp \
    -o test_length_prefix_framing

In [35]:
!./test_length_prefix_framing

Payload size: 29 bytes
Frame size: 33 bytes
Length-prefix framing test passed


## Step 5 — Batching Multiple Framed Events

The previous step added length-prefix framing for a single serialized profiling event:

```text
ProfileEvent
    ↓
serialize()
    ↓
payload
    ↓
frame()
    ↓
[length][payload]
```

The next step is to combine multiple framed events into one larger byte buffer.

This process is called:

```text
Batching
```

---

## Why Batch Multiple Events?

Without batching, each event may be handled independently:

```text
Event 1 → frame 1 → send()
Event 2 → frame 2 → send()
Event 3 → frame 3 → send()
Event 4 → frame 4 → send()
```

This is simple, but it may require many:

```text
function calls
system calls
network writes
```

If profiling events are generated at high frequency, this per-event overhead can become significant.

Instead, multiple framed events can be grouped together:

```text
Event 1 → frame 1 ┐
Event 2 → frame 2 ├──→ Batch Buffer
Event 3 → frame 3 │
Event 4 → frame 4 ┘
```

Then the entire batch can later be sent with fewer transport operations.

---

## Batch Layout

Each individual event is already framed as:

```text
[length][payload]
```

Therefore, a batch can simply concatenate multiple frames:

```text
[frame 1][frame 2][frame 3][frame 4]
```

Expanding this:

```text
[length1][payload1]
[length2][payload2]
[length3][payload3]
[length4][payload4]
```

Because each frame contains its own length prefix, the receiver can walk through the batch sequentially.

---

## Batch Construction

The batch builder receives multiple framed events:

```text
frame 1
frame 2
frame 3
```

and appends them into one:

```cpp
std::vector<std::byte>
```

Conceptually:

```text
empty batch
    ↓
append frame 1
    ↓
append frame 2
    ↓
append frame 3
    ↓
final batch buffer
```

The batch itself does not need to understand the contents of the profiling events.

It only stores framed byte sequences.

---

## Batch Parsing

The receiver must later recover each frame from the batch.

Suppose the batch contains:

```text
[length1][payload1][length2][payload2][length3][payload3]
```

The parser starts at offset 0.

It reads:

```text
length1
```

then skips:

```text
sizeof(length field) + length1
```

to locate the next frame.

The process repeats:

```text
read length
    ↓
extract payload
    ↓
advance offset
    ↓
repeat
```

until the end of the batch is reached.

---

## Parsing With an Offset

A batch parser needs to maintain a current position:

```cpp
std::size_t offset = 0;
```

After one frame is processed:

```text
offset += header_size + payload_size
```

The next frame begins at the new offset.

Conceptually:

```text
Batch Buffer
┌─────────────────────────────────────────┐
│ frame 1 │ frame 2 │ frame 3 │ frame 4 │
└─────────────────────────────────────────┘
     ↑
   offset
```

After frame 1:

```text
Batch Buffer
┌─────────────────────────────────────────┐
│ frame 1 │ frame 2 │ frame 3 │ frame 4 │
└─────────────────────────────────────────┘
               ↑
             offset
```

---

## Validation

The batch parser must check that every frame is complete.

For example, if a frame claims:

```text
payload length = 100
```

but only:

```text
20 bytes
```

remain in the batch, the batch is malformed.

The parser should reject incomplete data rather than reading beyond the valid buffer.

---

## End-to-End Data Flow

The complete data path now becomes:

```text
ProfileEvent
    ↓
serialize()
    ↓
payload
    ↓
frame()
    ↓
framed event
    ↓
batch builder
    ↓
batch buffer
```

The reverse path becomes:

```text
batch buffer
    ↓
batch parser
    ↓
individual frames
    ↓
unframe()
    ↓
payload
    ↓
deserialize()
    ↓
ProfileEvent
```

---

## Why Frame Before Batching?

Framing should happen before batching because each event needs a self-contained message boundary.

If events were concatenated without framing:

```text
payload1payload2payload3
```

the receiver would not know where one event ends and the next begins.

With framing:

```text
[length1][payload1][length2][payload2]
```

the receiver can recover every event independently.

---

## Target Components

This step can introduce a simple:

```cpp
BatchBuilder
```

responsible for:

```text
append framed event
retrieve batch buffer
clear batch
```

and a:

```cpp
BatchParser
```

responsible for:

```text
read batch buffer
extract each payload
return multiple serialized events
```

Conceptually:

```text
FrameCodec
    ↓
BatchBuilder
    ↓
Batch Buffer
    ↓
BatchParser
    ↓
FrameCodec
```

---

## First Correctness Test

The first test should create several profiling events:

```text
KernelEvent
MemoryEvent
KernelEvent
MemoryEvent
```

Then perform:

```text
events
    ↓
serialize
    ↓
frame
    ↓
batch
    ↓
parse batch
    ↓
extract frames
    ↓
unframe
    ↓
deserialize
```

The final reconstructed events should match the original events and preserve their order.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- What is batching?
- Why can batching reduce per-event overhead?
- Why should events be framed before batching?
- How are multiple frames represented inside one byte buffer?
- How does an offset-based batch parser work?
- How does the parser detect incomplete frames?
- Why is event ordering preserved?
- How will this batch format prepare the system for TCP transport?

In [36]:
%%writefile distributed-profiler/include/batch_builder.hpp

#pragma once

#include <cstddef>
#include <vector>

class BatchBuilder {
public:

    // Add a function to append one framed event.
    //
    // Input:
    //   const std::vector<std::byte>& frame
    //
    // Behavior:
    //   append all bytes from frame to the internal batch buffer.
    void append(
        const std::vector<std::byte>& frame
    );

    // Add a function that returns the current batch buffer.
    const std::vector<std::byte>&
    data() const;

    void clear();


    // Add a size() function that returns
    // the current batch size in bytes.
    std::size_t
    size() const;

private:
    // Store the batch bytes here.
    std::vector<std::byte> buffer_;
};

Writing distributed-profiler/include/batch_builder.hpp


In [37]:
%%writefile distributed-profiler/include/batch_parser.hpp

#pragma once

#include <cstddef>
#include <optional>
#include <vector>

class BatchParser {
public:
    // Add a parse() function.
    //
    // Input:
    //   const std::vector<std::byte>& batch
    //
    // Output:
    //   std::optional<
    //       std::vector<std::vector<std::byte>>
    //   >
    //
    // Each inner vector should contain one COMPLETE FRAME.
    //
    // If the batch is malformed:
    //   return std::nullopt;
    std::optional<
        std::vector<std::vector<std::byte>>
    >
    parse(
        const std::vector<std::byte>& batch
    ) const;
};

Writing distributed-profiler/include/batch_parser.hpp


In [38]:
%%writefile distributed-profiler/src/batch_builder.cpp

#include "batch_builder.hpp"

void BatchBuilder::append(
    const std::vector<std::byte>& frame
) {
    // Append all bytes from frame into buffer_.
    buffer_.insert(
        buffer_.end(),
        frame.begin(),
        frame.end()
    );
}


const std::vector<std::byte>&
BatchBuilder::data() const {
    return buffer_;
}


void BatchBuilder::clear() {
    buffer_.clear();
}


std::size_t
BatchBuilder::size() const {
    return buffer_.size();
}

Writing distributed-profiler/src/batch_builder.cpp


In [39]:
%%writefile distributed-profiler/src/batch_parser.cpp

#include "batch_parser.hpp"

#include <cstdint>
#include <cstring>
/*offset = 0
   ↓
read payload_size
   ↓
frame_size = 4 + payload_size
   ↓
check enough bytes remain
   ↓
extract complete frame
   ↓
frames.push_back(...)
   ↓
offset += frame_size
   ↓
repeat*/

std::optional<
    std::vector<std::vector<std::byte>>
>
BatchParser::parse(
    const std::vector<std::byte>& batch
) const {

    std::vector<std::vector<std::byte>> frames;

    std::size_t offset = 0;

    // Loop while offset < batch.size().
       while (offset < batch.size()) {

        // At least 4 bytes must remain for the length prefix.
        // Check that at least 4 bytes remain
        // for the uint32_t length prefix.
        //
        // If not:
        //     return std::nullopt;
        if (
            batch.size() - offset <
            sizeof(std::uint32_t)
        ) {
            return std::nullopt;
        }

        // Read payload_size from:
        //
        // batch.data() + offset
        //
        // using std::memcpy.
        std::uint32_t payload_size = 0;
        std::memcpy(
            &payload_size,
            batch.data() + offset,
            sizeof(std::uint32_t)
        );

        // Calculate total frame size:
        //
        // sizeof(std::uint32_t) + payload_size
        std::size_t frame_size =
            sizeof(std::uint32_t) +
            static_cast<std::size_t>(payload_size);


        // Check that enough bytes remain in the batch.
        //
        // If:
        //
        // offset + frame_size > batch.size()
        //
        // return std::nullopt;
        if (frame_size > batch.size() - offset) {
            return std::nullopt;
        }


        // Extract the COMPLETE frame:
        //
        // [length][payload]
        //
        // into a new std::vector<std::byte>.
        auto frame_begin =
            batch.begin() + offset;

        auto frame_end =
            frame_begin + frame_size;

        std::vector<std::byte> frame(
            frame_begin,
            frame_end
        );


        // Push that frame into frames.
        frames.push_back(
            std::move(frame)
        );


        // Advance offset by frame_size.
        offset += frame_size;}


    // After the loop finishes,
    // return frames.
    return frames;
}

Writing distributed-profiler/src/batch_parser.cpp


In [43]:
%%writefile distributed-profiler/tests/test_batching.cpp

#include "batch_builder.hpp"
#include "batch_parser.hpp"
#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_deserializer.hpp"
#include "simple_text_serializer.hpp"

#include <iostream>
#include <type_traits>
#include <variant>
#include <vector>

int main() {
    SimpleTextSerializer serializer;
    SimpleTextDeserializer deserializer;
    FrameCodec codec;

    BatchBuilder builder;
    BatchParser parser;

    // Create several ProfileEvents.
    // Put them into:
    //
    // std::vector<ProfileEvent> events;
    std::vector<ProfileEvent> events = {
    KernelEvent{1000, 200, 0, "gemm_kernel"},
    MemoryEvent{1500, 0, 4096, "H2D"},
    KernelEvent{2000, 150, 0, "softmax_kernel"},
    MemoryEvent{2300, 0, 8192, "D2H"}};


    // TODO 22:
    // For every event:
    //
    // serialize
    //    ↓
    // frame
    //    ↓
    // builder.append(frame)
    for (const auto& event : events) {
        auto payload = serializer.serialize(event);

        auto frame = codec.frame(payload);

        builder.append(frame);
    }

    // Print:
    //
    // number of original events
    // total batch size
    std::cout
        << "Original events: "
        << events.size()
        << '\n';

    std::cout
        << "Total batch size: "
        << builder.size()
        << " bytes\n";

    // Parse:
    //
    // builder.data()
    //
    // using BatchParser.
    auto parsed_frames =
        parser.parse(builder.data());

    // Check parse succeeded.
    //
    // If not:
    //   print error
    //   return 1;
    if (!parsed_frames.has_value()) {
        std::cerr
            << "Batch parsing failed\n";

        return 1;
    }

    // Check that the number of extracted frames
    // equals the number of original events.
    if (parsed_frames->size() != events.size()) {
        std::cerr
            << "Frame count does not match original event count\n";

        return 1;
    }
    std::cout
        << "Parsed frames: "
        << parsed_frames->size()
        << '\n';

    // For each extracted frame:
    //
    // unframe
    //    ↓
    // deserialize
    //
    // Verify both operations succeed.
    for (
        std::size_t i = 0;
        i < parsed_frames->size();
        ++i
    ) {
        auto payload =
            codec.unframe((*parsed_frames)[i]);



    // Print each reconstructed event,
    // or otherwise verify that order is preserved.
        if (!payload.has_value()) {
            std::cerr
                << "Unframing failed for frame "
                << i
                << '\n';

            return 1;}

        auto event =
            deserializer.deserialize(*payload);

        if (!event.has_value()) {
            std::cerr
                << "Deserialization failed for frame "
                << i
                << '\n';

            return 1;}
            std::cout
            << "Reconstructed event "
            << i
            << ": ";

        std::visit(
            [](const auto& e) {
                using T =
                    std::decay_t<decltype(e)>;

                if constexpr (
                    std::is_same_v<T, KernelEvent>
                ) {
                    std::cout
                        << "KernelEvent"
                        << " kernel=" << e.kernel_name
                        << " timestamp=" << e.timestamp_ns
                        << " duration=" << e.duration_ns
                        << " device=" << e.device_id;
                }
                else if constexpr (
                    std::is_same_v<T, MemoryEvent>
                ) {
                    std::cout
                        << "MemoryEvent"
                        << " operation=" << e.operation
                        << " timestamp=" << e.timestamp_ns
                        << " bytes=" << e.bytes
                        << " device=" << e.device_id;
                }
            },
            *event
        );

        std::cout << '\n';
    }
    // Print:
    //
    // Batch correctness test passed

    std::cout
        << "Batch correctness test passed\n";


    return 0;
}

Overwriting distributed-profiler/tests/test_batching.cpp


In [44]:
!g++ \
    -std=c++20 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/simple_text_deserializer.cpp \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/batch_builder.cpp \
    distributed-profiler/src/batch_parser.cpp \
    distributed-profiler/tests/test_batching.cpp \
    -o test_batching

In [45]:
!./test_batching

Original events: 4
Total batch size: 121 bytes
Parsed frames: 4
Reconstructed event 0: KernelEvent kernel=gemm_kernel timestamp=1000 duration=200 device=0
Reconstructed event 1: MemoryEvent operation=H2D timestamp=1500 bytes=4096 device=0
Reconstructed event 2: KernelEvent kernel=softmax_kernel timestamp=2000 duration=150 device=0
Reconstructed event 3: MemoryEvent operation=D2H timestamp=2300 bytes=8192 device=0
Batch correctness test passed


## Step 6 — Batch Size Benchmark and Tradeoff Analysis

The previous steps validated the complete correctness path:

```text
ProfileEvent
    ↓
serialize()
    ↓
frame()
    ↓
batch
    ↓
parse batch
    ↓
unframe()
    ↓
deserialize()
    ↓
ProfileEvent
```

The final step of this notebook focuses on performance rather than correctness.

The main question is:

```text
How does batch size affect throughput and batching overhead?
```

---

## Why Benchmark Batch Size?

Batching reduces per-event overhead by grouping multiple framed events into one larger buffer.

Without batching:

```text
event 1 → frame → flush
event 2 → frame → flush
event 3 → frame → flush
```

With batching:

```text
event 1 ┐
event 2 ├→ one batch → flush
event 3 ┘
```

A larger batch can reduce the number of flush operations and amortize fixed overhead across more events.

However, larger batches may also increase the time an individual event waits before the batch is ready.

Therefore, batch size introduces an important systems tradeoff:

```text
throughput
vs.
latency
```

---

## Controlled Workload

The benchmark should generate a large number of synthetic profiling events.

For example:

```text
100,000 events
```

The workload should remain identical for every experiment.

Only the batch size should change.

This allows the benchmark to isolate the effect of batching.

---

## Batch Size Sweep

The benchmark can test batch sizes such as:

```text
1
4
16
64
256
```

Here, batch size means:

```text
number of events per batch
```

For example:

```text
batch size = 1
```

means:

```text
[event]
→ one batch
```

while:

```text
batch size = 4
```

means:

```text
[event][event][event][event]
→ one batch
```

---

## Benchmark Pipeline

For every event, the benchmark performs:

```text
ProfileEvent
    ↓
serialize()
    ↓
frame()
    ↓
append to BatchBuilder
```

When the configured batch size is reached:

```text
batch full
    ↓
record batch statistics
    ↓
clear BatchBuilder
    ↓
continue
```

At the end, any remaining events should also be flushed.

---

## Metrics

For each batch size, the benchmark should measure:

```text
total events
total elapsed time
events per second
total bytes processed
MB per second
number of batches
average batch size in bytes
average bytes per event
```

A result table might look conceptually like:

```text
Batch Size | Time | Events/s | MB/s | Batches | Avg Batch Bytes
-----------|------|----------|------|---------|----------------
1          | ...  | ...      | ...  | ...     | ...
4          | ...  | ...      | ...  | ...     | ...
16         | ...  | ...      | ...  | ...     | ...
64         | ...  | ...      | ...  | ...     | ...
256        | ...  | ...      | ...  | ...     | ...
```

---

## Throughput

Throughput measures how much work the pipeline can process per unit time.

One useful metric is:

```text
events per second
```

Conceptually:

```text
events_per_second =
total_events / elapsed_time
```

Another useful metric is:

```text
MB per second
```

which measures serialized data throughput.

---

## Number of Batches

As batch size increases, the number of generated batches should decrease.

For example, with:

```text
1000 events
```

and:

```text
batch size = 1
```

approximately:

```text
1000 batches
```

are generated.

With:

```text
batch size = 100
```

approximately:

```text
10 batches
```

are generated.

This is one reason batching can reduce per-batch overhead.

---

## Expected Trend

A small batch size usually means:

```text
more batches
more per-batch overhead
lower batching delay
```

A larger batch size usually means:

```text
fewer batches
better overhead amortization
potentially higher throughput
higher potential buffering delay
```

The expected tradeoff is therefore:

```text
small batch
    ↓
lower latency
higher overhead

large batch
    ↓
higher throughput
potentially higher latency
```

---

## Important Benchmark Limitation

In this notebook, the benchmark measures:

```text
serialization
framing
batch construction
```

but does not yet include real network transmission.

Therefore, the measured results represent:

```text
host-side batching performance
```

rather than complete distributed-system throughput.

The next notebook will introduce TCP transport, where batching should have a stronger effect because it can also reduce the number of network send operations.

---

## Goals of This Step

By the end of this section, the following questions should be answerable:

- Why benchmark different batch sizes?
- How does batch size affect the number of batches?
- Why can larger batches improve throughput?
- What does it mean to amortize fixed overhead?
- What is the difference between events per second and MB per second?
- Why can larger batches increase buffering latency?
- Why should only one variable change during a controlled benchmark?
- What are the limitations of a host-only batching benchmark?
- How does this experiment prepare the system for TCP transport?

In [46]:
%%writefile distributed-profiler/benchmarks/benchmark_batch_sizes.cpp

#include "batch_builder.hpp"
#include "frame_codec.hpp"
#include "profile_event.hpp"
#include "simple_text_serializer.hpp"

#include <chrono>
#include <cstddef>
#include <iomanip>
#include <iostream>
#include <vector>

int main() {
    SimpleTextSerializer serializer;
    FrameCodec codec;

    const std::size_t total_events = 100000;

    std::vector<std::size_t> batch_sizes = {
        1, 4, 16, 64, 256
    };

    std::vector<ProfileEvent> events;
    events.reserve(total_events);

    for (std::size_t i = 0; i < total_events; ++i) {
        if (i % 2 == 0) {
            events.push_back(
                KernelEvent{
                    1000 + i,
                    200,
                    0,
                    "gemm_kernel"
                }
            );
        }
        else {
            events.push_back(
                MemoryEvent{
                    1000 + i,
                    0,
                    4096,
                    "H2D"
                }
            );
        }
    }

    std::cout
        << std::left
        << std::setw(12) << "BatchSize"
        << std::setw(14) << "Time(ms)"
        << std::setw(16) << "Events/s"
        << std::setw(12) << "MB/s"
        << std::setw(12) << "Batches"
        << std::setw(18) << "AvgBatchBytes"
        << '\n';

    for (std::size_t batch_size : batch_sizes) {
        BatchBuilder builder;

        std::size_t events_in_batch = 0;
        std::size_t number_of_batches = 0;
        std::size_t total_batch_bytes = 0;

        auto start =
            std::chrono::steady_clock::now();

        for (const auto& event : events) {
            auto payload =
                serializer.serialize(event);

            auto frame =
                codec.frame(payload);

            builder.append(frame);

            ++events_in_batch;

            if (events_in_batch == batch_size) {
                total_batch_bytes += builder.size();

                ++number_of_batches;

                builder.clear();

                events_in_batch = 0;
            }
        }

        if (builder.size() > 0) {
            total_batch_bytes += builder.size();

            ++number_of_batches;

            builder.clear();
        }

        auto end =
            std::chrono::steady_clock::now();

        double elapsed_seconds =
            std::chrono::duration<double>(
                end - start
            ).count();

        double elapsed_ms =
            elapsed_seconds * 1000.0;

        double events_per_second =
            static_cast<double>(total_events) /
            elapsed_seconds;

        double total_mb =
            static_cast<double>(total_batch_bytes) /
            (1024.0 * 1024.0);

        double mb_per_second =
            total_mb /
            elapsed_seconds;

        double average_batch_bytes =
            static_cast<double>(total_batch_bytes) /
            static_cast<double>(number_of_batches);

        std::cout
            << std::left
            << std::setw(12) << batch_size
            << std::setw(14) << std::fixed << std::setprecision(3)
            << elapsed_ms
            << std::setw(16) << std::setprecision(0)
            << events_per_second
            << std::setw(12) << std::setprecision(2)
            << mb_per_second
            << std::setw(12)
            << number_of_batches
            << std::setw(18)
            << average_batch_bytes
            << '\n';
    }

    return 0;
}

Writing distributed-profiler/benchmarks/benchmark_batch_sizes.cpp


In [47]:
!g++ \
    -std=c++20 \
    -O2 \
    -Wall \
    -Wextra \
    -Wpedantic \
    -Idistributed-profiler/include \
    distributed-profiler/src/simple_text_serializer.cpp \
    distributed-profiler/src/frame_codec.cpp \
    distributed-profiler/src/batch_builder.cpp \
    distributed-profiler/benchmarks/benchmark_batch_sizes.cpp \
    -o benchmark_batch_sizes

In [48]:
!./benchmark_batch_sizes

BatchSize   Time(ms)      Events/s        MB/s        Batches     AvgBatchBytes     
1           26.490        3774964         109.51      100000      30.42             
4           26.778        3734451         108.34      25000       121.68            
16          26.478        3776674         109.56      6250        486.72            
64          38.142        2621814         76.06       1563        1946.26           
256         32.408        3085692         89.52       391         7780.05           
